Model Caller

In [1]:
import math
import json
import os
import numpy as np
import librosa
import joblib
from tensorflow.keras import models

def load_emotion_artifacts(model_path, scaler_path):
    model = models.load_model(model_path)
    scaler = joblib.load(scaler_path)
    return model, scaler


def predict_emotion_chunks(
    audio_path,
    model_path,
    scaler_path,
    chunk_secs=20
):
    """
    Predict one emotion label per chunk of audio.

    Returns:
        [
            {
                "start": 0.0,
                "end": 20.0,
                "emotion": "calm",
                "valence": 6.21,
                "arousal": 3.77
            },
            ...
        ]
    """
    model, scaler = load_emotion_artifacts(model_path, scaler_path)

    y_audio, sr = librosa.load(audio_path, sr=SR, mono=True)
    chunk_len = int(chunk_secs * sr)

    results = []

    for start_idx in range(0, len(y_audio), chunk_len):
        chunk = y_audio[start_idx:start_idx + chunk_len]

        # skip very tiny tail chunk
        if len(chunk) < chunk_len // 2:
            continue

        seg_len = int(SEGMENT_SECS * sr)
        segments = [
            chunk[i:i + seg_len]
            for i in range(0, len(chunk), seg_len)
            if len(chunk[i:i + seg_len]) == seg_len
        ]

        if not segments:
            continue

        mel_list = []
        for seg in segments:
            mel = librosa.feature.melspectrogram(
                y=seg,
                sr=sr,
                n_mels=N_MELS,
                n_fft=N_FFT,
                hop_length=HOP_LENGTH
            )
            mel_db = librosa.power_to_db(mel, ref=np.max)

            if mel_db.shape[1] < SEG_FRAMES:
                mel_db = np.pad(
                    mel_db,
                    ((0, 0), (0, SEG_FRAMES - mel_db.shape[1]))
                )
            else:
                mel_db = mel_db[:, :SEG_FRAMES]

            mel_list.append(mel_db)

        X = np.expand_dims(np.array(mel_list, dtype=np.float32), -1)
        preds_n = model.predict(X, verbose=0)

        mean_orig = scaler.inverse_transform([preds_n.mean(axis=0)])[0]
        valence = float(mean_orig[0])
        arousal = float(mean_orig[1])
        emotion = assign_emotion(valence, arousal)

        results.append({
            "start": round(start_idx / sr, 3),
            "end": round(min((start_idx + chunk_len) / sr, len(y_audio) / sr), 3),
            "emotion": emotion,
            "valence": round(valence, 3),
            "arousal": round(arousal, 3),
        })

    return results


def save_emotion_chunks(
    audio_path,
    model_path,
    scaler_path,
    output_json="emotion_chunks.json",
    chunk_secs=20
):
    chunks = predict_emotion_chunks(
        audio_path=audio_path,
        model_path=model_path,
        scaler_path=scaler_path,
        chunk_secs=chunk_secs
    )

    with open(output_json, "w") as f:
        json.dump(chunks, f, indent=2)

    return chunks

In [2]:
import os
import json
import librosa
import numpy as np
import cv2
from moviepy.editor import AudioFileClip, VideoFileClip

MOOD_TO_COLOR = {
    "happy":   (0, 215, 255),
    "content": (80, 200, 255),
    "calm":    (220, 180, 80),
    "relaxed": (200, 220, 120),
    "excited": (0, 140, 255),
    "tense":   (30, 30, 220),
    "gloomy":  (120, 80, 140),
    "sad":     (180, 90, 40),
    "neutral": (160, 160, 160)
}

# MOOD_TO_SHAPE is no longer used directly for shape type, but kept for context if needed for other visual elements
MOOD_TO_SHAPE = {
    "happy":   0,   # circle
    "content": 5,   # flower
    "calm":    2,   # polygon
    "relaxed": 5,   # flower
    "excited": 4,   # burst
    "tense":   1,   # star
    "gloomy":  3,   # spiral
    "sad":     2,   # polygon
    "neutral": 0
}

def emotion_at_time(t, emotion_chunks):
    for chunk in emotion_chunks:
        if chunk["start"] <= t < chunk["end"]:
            return chunk["emotion"]
    return emotion_chunks[-1]["emotion"] if emotion_chunks else "neutral"

def assign_emotion(valence, arousal):
    """Assigns an emotion label based on valence and arousal values (1-9 scale)."""
    # Assuming valence and arousal are on a 1-9 scale, with 5 being neutral.
    # Adjust these thresholds and labels as needed for specific emotion models.

    if valence >= 6.5 and arousal >= 6.5:
        return "excited"
    elif valence >= 6.5 and arousal < 6.5 and arousal >= 3.5:
        return "happy"
    elif valence >= 6.5 and arousal < 3.5:
        return "relaxed"
    elif valence < 3.5 and arousal >= 6.5:
        return "tense"
    elif valence < 3.5 and arousal < 6.5 and arousal >= 3.5:
        return "sad"
    elif valence < 3.5 and arousal < 3.5:
        return "gloomy"
    elif valence >= 3.5 and valence < 6.5 and arousal >= 6.5:
        return "content"
    elif valence >= 3.5 and valence < 6.5 and arousal < 3.5:
        return "calm"
    else:
        return "neutral"

# This function is being removed as the shape type will be calculated directly based on audio features for more variety.
# def shape_from_mood_and_audio(mood, z, tr):
#     base = MOOD_TO_SHAPE.get(mood, 0)

#     if mood in ["happy", "content", "relaxed"]:
#         return [base, 5, 0][int((z + tr) * 1.5) % 3]
#     elif mood in ["tense", "excited"]:
#         return [base, 1, 4][int((z + tr) * 1.5) % 3]
#     elif mood in ["sad", "gloomy", "calm"]:
#         return [base, 2, 3][int((z + tr) * 1.5) % 3]

#     return base

def norm(x):
    return (x - x.min()) / (x.max() - x.min()) if x.max() != x.min() else np.zeros_like(x)

def draw_shape(frame, s):
    overlay = frame.copy()

    x, y = s["x"], s["y"]
    size = s["size"]
    color = s["color"]
    angle = s["angle"]
    t = s["type"] % 6

    if t == 0:
        cv2.circle(overlay, (x, y), size, color, -1)

    elif t == 1:
        pts = []
        for i in range(10):
            a = i * np.pi / 5 + angle
            rr = size if i % 2 == 0 else size // 2
            pts.append((int(x + rr * np.cos(a)), int(y + rr * np.sin(a))))
        cv2.fillPoly(overlay, [np.array(pts)], color)

    elif t == 2:
        sides = 3 + (size % 5)
        pts = []
        for i in range(sides):
            a = i * 2 * np.pi / sides + angle
            pts.append((int(x + size * np.cos(a)), int(y + size * np.sin(a))))
        cv2.fillPoly(overlay, [np.array(pts)], color)

    elif t == 3:
        for i in range(20):
            rr = size * i / 20
            a = angle + i * 0.3
            px = int(x + rr * np.cos(a))
            py = int(y + rr * np.sin(a))
            cv2.circle(overlay, (px, py), 2, color, -1)

    elif t == 4:
        for i in range(12):
            a = i * 2 * np.pi / 12 + angle
            px = int(x + size * np.cos(a))
            py = int(y + size * np.sin(a))
            cv2.line(overlay, (x, y), (px, py), color, 2)

    else:
        pts = []
        for i in range(100):
            a = i * 2 * np.pi / 100
            rr = size * (1 + 0.3 * np.sin(5 * a))
            pts.append((int(x + rr * np.cos(a)), int(y + rr * np.sin(a))))
        cv2.fillPoly(overlay, [np.array(pts)], color)

    cv2.addWeighted(overlay, s["alpha"], frame, 1 - s["alpha"], 0, frame)

def render_music_video(audio_path, emotion_chunks, output_video="final_fixed.mp4", fps=20, frame_size=640):
    # ======================
    # LOAD AUDIO
    # ======================
    y, sr = librosa.load(audio_path, sr=22050)
    duration = len(y) / sr

    # ======================
    # ONSETS
    # ======================
    onsets = librosa.onset.onset_detect(y=y, sr=sr, units="time")

    # ======================
    # FEATURES
    # ======================
    hop = 512

    rms = librosa.feature.rms(y=y, hop_length=hop).flatten()
    centroid = librosa.feature.spectral_centroid(y=y, sr=sr, hop_length=hop).flatten()
    zcr = librosa.feature.zero_crossing_rate(y=y, hop_length=hop).flatten()

    S = np.abs(librosa.stft(y))
    freqs = librosa.fft_frequencies(sr=sr)

    bass = S[(freqs < 200)].mean(axis=0)
    treble = S[(freqs >= 2000)].mean(axis=0)

    rms = norm(rms)
    centroid = norm(centroid)
    zcr = norm(zcr)
    bass = norm(bass)
    treble = norm(treble)

    times = np.arange(len(rms)) * hop / sr

    def interp(f, t):
        return np.interp(t, times, f)

    durations = np.diff(onsets, append=duration)

    # ======================
    # MAP SHAPES
    # ======================
    shapes = []

    for i, t in enumerate(onsets):
        r = interp(rms, t)
        c = interp(centroid, t)
        z = interp(zcr, t)
        b = interp(bass, t)
        tr = interp(treble, t)
        dur = durations[i]

        mood = emotion_at_time(t, emotion_chunks)
        # Combine mood-based base shape with audio features for variety
        base_shape = MOOD_TO_SHAPE.get(mood, 0)
        shape_type = (base_shape + int((z + b + tr) * 6)) % 6

        x = np.clip((r**0.5 + np.random.uniform(-0.1, 0.1)), 0.05, 0.95)
        y_pos = np.clip((c**0.5 + np.random.uniform(-0.1, 0.1)), 0.05, 0.95)

        shapes.append({
            "time": t,
            "x": int(x * frame_size),
            "y": int(y_pos * frame_size),
            "size": int(8 + r * 50),
            "color": (
                int(255 * c),
                int(255 * b),
                int(255 * (1 - c))
            ),
            "angle": tr * 2 * np.pi,
            "type": shape_type,
            "alpha": float(np.clip(0.1 + 0.7 * r + 0.3 * np.random.rand() * dur, 0.1, 1.0))
        })

    # ======================
    # RENDER SILENT VIDEO
    # ======================
    temp_video = "temp_silent.mp4"

    out = cv2.VideoWriter(
        temp_video,
        cv2.VideoWriter_fourcc(*"mp4v"),
        fps,
        (frame_size, frame_size)
    )

    canvas = np.zeros((frame_size, frame_size, 3), dtype=np.uint8)
    shape_index = 0
    total_frames = int(duration * fps)

    for i in range(total_frames):
        t = i / fps

        while shape_index < len(shapes) and shapes[shape_index]["time"] <= t:
            draw_shape(canvas, shapes[shape_index])
            shape_index += 1

        out.write(canvas)

    out.release()

    # ======================
    # ADD AUDIO
    # ======================
    video = VideoFileClip(temp_video)
    audio = AudioFileClip(audio_path)
    final = video.set_audio(audio)
    final.write_videofile(output_video, codec="libx264", audio_codec="aac")

    video.close()
    audio.close()
    final.close()

    if os.path.exists(temp_video):
        os.remove(temp_video)

    return output_video

/usr/local/lib/python3.12/dist-packages/moviepy/config_defaults.py:47: SyntaxWarning: invalid escape sequence '\P'
  IMAGEMAGICK_BINARY = r"C:\Program Files\ImageMagick-6.8.8-Q16\magick.exe"
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:294: SyntaxWarning: invalid escape sequence '\d'
  lines_video = [l for l in lines if ' Video: ' in l and re.search('\d+x\d+', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:367: SyntaxWarning: invalid escape sequence '\d'
  rotation_lines = [l for l in lines if 'rotate          :' in l and re.search('\d+$', l)]
/usr/local/lib/python3.12/dist-packages/moviepy/video/io/ffmpeg_reader.py:370: SyntaxWarning: invalid escape sequence '\d'
  match = re.search('\d+$', rotation_line)
  if event.key is 'enter':



RUN PIPELINE

In [3]:
import argparse
import os

def run_pipeline(audio_path, model_path, scaler_path, output_path, chunk_secs=20):
    print(f"\nProcessing: {audio_path}")

    emotion_chunks = predict_emotion_chunks(
        audio_path=audio_path,
        model_path=model_path,
        scaler_path=scaler_path,
        chunk_secs=chunk_secs
    )

    print("\nEmotion timeline:")
    for c in emotion_chunks:
        print(c)

    render_music_video(
        audio_path=audio_path,
        emotion_chunks=emotion_chunks,
        output_video=output_path
    )

    print(f"\nSaved output → {output_path}")

Run all cells then change paths below

In [5]:
audio_path = "CityOfStars.mp3"
model_path = "/content/best_cnn_model.keras"
scaler_path = "/content/label_scaler.pkl"
output_path = "CityOfStars.mp4"

# Define global constants for audio processing and model compatibility (SR and others are defined here)
global SR, N_MELS, N_FFT, HOP_LENGTH, SEGMENT_SECS, SEG_FRAMES
SR = 22050       # Sample rate
N_MELS = 128     # Number of Mel bands
N_FFT = 2048     # FFT window size
HOP_LENGTH = 512 # Hop length for FFT
SEGMENT_SECS = 10  # Adjust segment duration to yield ~431 frames
SEG_FRAMES = 431   # Model expects this number of frames

run_pipeline(audio_path, model_path, scaler_path, output_path, chunk_secs=20)


Processing: CityOfStars.mp3

Emotion timeline:
{'start': 0.0, 'end': 20.0, 'emotion': 'neutral', 'valence': 4.419, 'arousal': 4.179}
{'start': 20.0, 'end': 40.0, 'emotion': 'neutral', 'valence': 4.312, 'arousal': 4.139}
{'start': 40.0, 'end': 60.0, 'emotion': 'neutral', 'valence': 4.709, 'arousal': 4.274}
{'start': 60.0, 'end': 80.0, 'emotion': 'neutral', 'valence': 4.799, 'arousal': 4.246}
{'start': 80.0, 'end': 100.0, 'emotion': 'excited', 'valence': 6.922, 'arousal': 6.802}
{'start': 100.0, 'end': 120.0, 'emotion': 'neutral', 'valence': 5.866, 'arousal': 5.408}
{'start': 120.0, 'end': 140.0, 'emotion': 'neutral', 'valence': 3.877, 'arousal': 3.817}
Moviepy - Building video CityOfStars.mp4.
MoviePy - Writing audio in CityOfStarsTEMP_MPY_wvf_snd.mp4


MoviePy - Done.
Moviepy - Writing video CityOfStars.mp4



Moviepy - Done !
Moviepy - video ready CityOfStars.mp4

Saved output → CityOfStars.mp4


In [ ]:
from google.colab import drive
drive.mount('/content/drive')